# Population Divergence and Admixture Simulation

This notebook simulates an ancestral population that splits into two daughter populations, which later contribute to an admixed population. The simulation uses SLiM5 to model a recombining genome and outputs a tree sequence that is analyzed using tskit.

## Model Description

1. **Ancestral Population**: A population of size `Na` exists from the start
2. **Population Split**: At time `Tsplit`, the ancestral population splits into two daughter populations of sizes `N1` and `N2`
3. **Divergence**: The two populations diverge without gene flow until time `Tadmix`
4. **Admixture**: At time `Tadmix`, a third population of size `NAdmix` is created by merging a fraction `f` from population 1 and `(1-f)` from population 2


## Requirements

- **SLiM**: Version 4.0 or later (tested with SLiM 4.0+)
- **Python**: Version 3.8 or later
- **Python packages**: tskit, numpy, matplotlib, pandas
## Parameters

- `Na`: Ancestral population size
- `N1`: Size of daughter population 1
- `N2`: Size of daughter population 2
- `NAdmix`: Size of admixed population
- `Tsplit`: Generation at which population split occurs
- `Tadmix`: Generation at which admixture occurs
- `f`: Admixture fraction from population 1 (1-f from population 2)
- `genome_length`: Length of simulated genome in base pairs
- `recomb_rate`: Recombination rate per base pair per generation
- `mutation_rate`: Mutation rate per base pair per generation

In [ ]:
# Import required libraries
import subprocess
import os
import tempfile
import tskit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Display settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## Define Simulation Parameters

In [ ]:
# Population sizes
Na = 1000       # Ancestral population size
N1 = 800        # Daughter population 1 size
N2 = 1200       # Daughter population 2 size
NAdmix = 1000   # Admixed population size

# Timing (in generations)
# Note: We use recapitation to simulate the burn-in period for mutation-drift equilibrium
Tburn = 10 * Na  # Burn-in time for ancestral population (10N generations)
Tsplit = 1000   # Generation at which split occurs (after burn-in)
Tadmix = 2000   # Generation at which admixture occurs (after burn-in)
Tend = 2500     # Final generation (after burn-in)

# Admixture proportion
f = 0.6         # Fraction from population 1 (0.4 from population 2)

# Genome parameters
genome_length = 1e6      # 1 Mb genome
recomb_rate = 1e-8       # Recombination rate per bp per generation
mutation_rate = 1e-8     # Mutation rate per bp per generation

# Output file
output_trees = "simulation_output.trees"

print("Simulation Parameters:")
print(f"  Ancestral pop size (Na): {Na}")
print(f"  Burn-in time (Tburn): {Tburn} generations ({Tburn/Na:.1f}N)")
print(f"  Pop 1 size (N1): {N1}")
print(f"  Pop 2 size (N2): {N2}")
print(f"  Admixed pop size (NAdmix): {NAdmix}")
print(f"  Split time (Tsplit): {Tsplit} generations after burn-in")
print(f"  Admixture time (Tadmix): {Tadmix} generations after burn-in")
print(f"  End time (Tend): {Tend} generations after burn-in")
print(f"  Admixture fraction from pop1 (f): {f}")
print(f"  Genome length: {genome_length:.0e} bp")
print(f"  Recombination rate: {recomb_rate:.0e} per bp per gen")
print(f"  Mutation rate: {mutation_rate:.0e} per bp per gen")

## Generate SLiM5 Script

The following cell creates the SLiM5 script that implements the demographic model.

In [ ]:
# Create SLiM script
# Note: We do NOT add mutations during SLiM simulation
# Instead, we will use recapitation and add mutations afterward in tskit
slim_script = f"""
initialize() {{
    initializeTreeSeq();
    initializeMutationRate(0.0);  // No mutations during forward simulation
    initializeMutationType("m1", 0.5, "f", 0.0);  // Neutral mutations (not used)
    initializeGenomicElementType("g1", m1, 1.0);
    initializeGenomicElement(g1, 0, {int(genome_length)-1});
    initializeRecombinationRate({recomb_rate});
}}

// Create ancestral population
1 {{
    sim.addSubpop("p0", {Na});
}}

// Split into two populations at Tsplit
{Tsplit} {{
    sim.addSubpopSplit("p1", {N1}, p0);
    sim.addSubpopSplit("p2", {N2}, p0);
    p0.setSubpopulationSize(0);  // Remove ancestral population
}}

// Create admixed population at Tadmix
{Tadmix} {{
    sim.addSubpop("p3", {NAdmix});
    p3.setMigrationRates(c(p1, p2), c({f}, {1-f}));
}}

// Stop migration after one generation (admixture pulse)
{Tadmix + 1} {{
    p3.setMigrationRates(c(p1, p2), c(0.0, 0.0));
}}

// End simulation and output tree sequence
{Tend} late() {{
    sim.treeSeqOutput("{output_trees}");
    sim.simulationFinished();
}}
"""

# Save SLiM script to file
slim_script_file = "divergence_admixture.slim"
with open(slim_script_file, 'w') as f:
    f.write(slim_script)

print(f"SLiM script saved to: {slim_script_file}")
print("\nScript preview:")
print("=" * 60)
print(slim_script)
print("=" * 60)
print("\nNote: Mutations will be added after recapitation in tskit")

## Run SLiM Simulation

Execute the SLiM script to generate the tree sequence.

In [ ]:
# Run SLiM simulation
print("Running SLiM simulation...")
print("This may take a few moments depending on the parameter values.\n")

try:
    result = subprocess.run(
        ['slim', slim_script_file],
        capture_output=True,
        text=True,
        timeout=300  # 5 minute timeout
    )
    
    if result.returncode == 0:
        print("✓ Simulation completed successfully!")
        if result.stdout:
            print("\nSLiM output:")
            print(result.stdout)
    else:
        print("✗ Simulation failed!")
        print("Error output:")
        print(result.stderr)
        raise RuntimeError("SLiM simulation failed")
        
except FileNotFoundError:
    print("✗ Error: SLiM executable not found!")
    print("Please ensure SLiM is installed and in your PATH.")
    print("You can install SLiM from: https://messerlab.org/slim/")
    raise
except subprocess.TimeoutExpired:
    print("✗ Error: Simulation timed out!")
    print("Consider reducing population sizes or genome length.")
    raise

# Verify output file exists
if os.path.exists(output_trees):
    file_size = os.path.getsize(output_trees)
    print(f"\n✓ Tree sequence file created: {output_trees}")
    print(f"  File size: {file_size / 1024:.2f} KB")
else:
    print(f"\n✗ Error: Output file {output_trees} was not created!")
    raise FileNotFoundError(f"Expected output file {output_trees} not found")

## Load and Inspect Tree Sequence

Load the tree sequence using tskit and examine its basic properties.

In [ ]:
# Load tree sequence
ts = tskit.load(output_trees)

print("Tree Sequence Summary")
print("=" * 60)
print(f"Sequence length: {ts.sequence_length:,.0f} bp")
print(f"Number of trees: {ts.num_trees:,}")
print(f"Number of samples: {ts.num_samples:,}")
print(f"Number of nodes: {ts.num_nodes:,}")
print(f"Number of mutations: {ts.num_mutations:,}")
print(f"Number of populations: {ts.num_populations}")
print(f"Number of individuals: {ts.num_individuals:,}")

# Display population information
print("\nPopulation Information:")
print("-" * 60)
for pop in ts.populations():
    print(f"Population {pop.id}: {pop.metadata}")

# Count samples per population
print("\nSamples per population:")
print("-" * 60)
for pop_id in range(ts.num_populations):
    sample_nodes = [node.id for node in ts.nodes() if node.population == pop_id and node.is_sample()]
    print(f"Population {pop_id}: {len(sample_nodes)} samples")

## Recapitate and Add Mutations

To ensure the ancestral population has reached mutation-drift equilibrium, we use tskit's `recapitate()` function to simulate the coalescent history before the start of the forward simulation.

Then we add mutations explicitly using the `mutate()` function, which allows us to track when each mutation arose.

In [ ]:
# Recapitate to add coalescent history before the forward simulation
print("Recapitating tree sequence...")
print(f"  Adding {Tburn} generations of coalescent history")
print(f"  Ancestral population size: {Na}")

# Recapitate with appropriate parameters
ts_recap = tskit.recapitate(
    ts,
    recombination_rate=recomb_rate,
    population_size=Na,
    random_seed=42
)

print(f"\n✓ Recapitation complete")
print(f"  Original trees: {ts.num_trees:,}")
print(f"  After recapitation: {ts_recap.num_trees:,}")

# Add mutations to the tree sequence
print("\nAdding mutations...")
ts_mutated = tskit.mutate(
    ts_recap,
    rate=mutation_rate,
    random_seed=42,
    keep=False  # Don't keep existing mutations (there shouldn't be any)
)

print(f"\n✓ Mutations added")
print(f"  Number of mutations: {ts_mutated.num_mutations:,}")
print(f"  Number of sites: {ts_mutated.num_sites:,}")

# Store the split time in generations from the start (including recapitation)
split_time_total = Tsplit

print(f"\n✓ Tree sequence ready for analysis")
print(f"  Split occurred at generation {Tsplit} (forward time)")

# Update ts to use the mutated version for all downstream analyses
ts = ts_mutated

## Analyze Population Structure

Calculate genetic diversity statistics for each population.

In [ ]:
# Get sample nodes for each population
pop_samples = {}
for pop_id in range(ts.num_populations):
    pop_samples[pop_id] = [node.id for node in ts.nodes() if node.population == pop_id and node.is_sample()]

# Calculate diversity statistics
print("Genetic Diversity Statistics")
print("=" * 60)

diversity_stats = []

for pop_id, samples in pop_samples.items():
    if len(samples) >= 2:
        # Nucleotide diversity (π)
        pi = ts.diversity(sample_sets=[samples], mode='site')
        
        # Tajima's D
        try:
            tajd = ts.Tajimas_D(sample_sets=[samples], mode='site')
        except:
            tajd = np.nan
        
        diversity_stats.append({
            'Population': pop_id,
            'Samples': len(samples),
            'Pi': pi[0] if len(pi) > 0 else np.nan,
            'Tajimas_D': tajd[0] if hasattr(tajd, '__len__') and len(tajd) > 0 else tajd
        })
        
        print(f"Population {pop_id}:")
        print(f"  Samples: {len(samples)}")
        print(f"  Nucleotide diversity (π): {pi[0] if len(pi) > 0 else 'N/A'}")
        print(f"  Tajima's D: {tajd[0] if hasattr(tajd, '__len__') and len(tajd) > 0 else tajd}")
        print()

# Create DataFrame
if diversity_stats:
    df_diversity = pd.DataFrame(diversity_stats)
    print("\nSummary Table:")
    print(df_diversity.to_string(index=False))

## Calculate Pairwise Divergence (Dxy)

Calculate divergence between population pairs to examine the effects of split and admixture.

In [ ]:
# Calculate pairwise divergence between populations
print("Pairwise Divergence (Dxy) Between Populations")
print("=" * 60)

divergence_matrix = np.zeros((ts.num_populations, ts.num_populations))

for pop1_id in range(ts.num_populations):
    for pop2_id in range(pop1_id, ts.num_populations):
        samples1 = pop_samples.get(pop1_id, [])
        samples2 = pop_samples.get(pop2_id, [])
        
        if len(samples1) >= 1 and len(samples2) >= 1:
            if pop1_id == pop2_id:
                # Within-population diversity
                dxy = ts.diversity(sample_sets=[samples1], mode='site')
            else:
                # Between-population divergence
                dxy = ts.divergence(sample_sets=[samples1, samples2], mode='site')
            
            div_value = dxy[0] if len(dxy) > 0 else 0
            divergence_matrix[pop1_id, pop2_id] = div_value
            divergence_matrix[pop2_id, pop1_id] = div_value
            
            if pop1_id != pop2_id:
                print(f"Pop {pop1_id} vs Pop {pop2_id}: {div_value:.6f}")

# Visualize divergence matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(divergence_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(ts.num_populations))
ax.set_yticks(range(ts.num_populations))
ax.set_xlabel('Population')
ax.set_ylabel('Population')
ax.set_title('Pairwise Genetic Divergence (Dxy)')
plt.colorbar(im, ax=ax, label='Divergence')

# Add text annotations
for i in range(ts.num_populations):
    for j in range(ts.num_populations):
        text = ax.text(j, i, f'{divergence_matrix[i, j]:.4f}',
                      ha="center", va="center", color="black", fontsize=10)

plt.tight_layout()
plt.show()

## Calculate FST Between Populations

FST measures population differentiation and helps quantify the genetic distance between populations.

In [ ]:
# Calculate FST between population pairs
print("Pairwise FST Between Populations")
print("=" * 60)

fst_matrix = np.zeros((ts.num_populations, ts.num_populations))

for pop1_id in range(ts.num_populations):
    for pop2_id in range(pop1_id + 1, ts.num_populations):
        samples1 = pop_samples.get(pop1_id, [])
        samples2 = pop_samples.get(pop2_id, [])
        
        if len(samples1) >= 2 and len(samples2) >= 2:
            fst = ts.Fst(sample_sets=[samples1, samples2], mode='site')
            fst_value = fst[0] if len(fst) > 0 else 0
            fst_matrix[pop1_id, pop2_id] = fst_value
            fst_matrix[pop2_id, pop1_id] = fst_value
            
            print(f"Pop {pop1_id} vs Pop {pop2_id}: FST = {fst_value:.6f}")

# Visualize FST matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(fst_matrix, cmap='viridis', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(ts.num_populations))
ax.set_yticks(range(ts.num_populations))
ax.set_xlabel('Population')
ax.set_ylabel('Population')
ax.set_title('Pairwise FST Between Populations')
plt.colorbar(im, ax=ax, label='FST')

# Add text annotations
for i in range(ts.num_populations):
    for j in range(ts.num_populations):
        if i != j:
            text = ax.text(j, i, f'{fst_matrix[i, j]:.4f}',
                          ha="center", va="center", color="white", fontsize=10)

plt.tight_layout()
plt.show()

## Visualize Site Frequency Spectrum

The site frequency spectrum (SFS) shows the distribution of allele frequencies in each population.

In [ ]:
# Calculate and plot site frequency spectrum for each population
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for pop_id, samples in pop_samples.items():
    if len(samples) >= 2 and pop_id < 4:
        # Calculate allele frequency spectrum
        afs = ts.allele_frequency_spectrum(
            sample_sets=[samples],
            mode='site',
            polarised=True
        )
        
        # Plot SFS (excluding monomorphic sites)
        ax = axes[pop_id]
        x = np.arange(1, len(afs))  # Exclude 0 frequency
        y = afs[1:]  # Exclude monomorphic sites
        
        ax.bar(x, y, color=f'C{pop_id}', alpha=0.7)
        ax.set_xlabel('Derived Allele Count')
        ax.set_ylabel('Number of Sites')
        ax.set_title(f'Site Frequency Spectrum - Population {pop_id}\n({len(samples)} samples)')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)

# Remove extra subplot if fewer than 4 populations
for i in range(len(pop_samples), 4):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Visualize Tree Sequence

Draw a representation of the first few trees in the sequence to visualize genealogical relationships.

In [ ]:
# Simplify tree sequence for visualization (sample a subset if needed)
# Take up to 20 samples per population for clearer visualization
viz_samples = []
for pop_id, samples in pop_samples.items():
    viz_samples.extend(samples[:min(20, len(samples))])

# Draw the first tree
if ts.num_trees > 0:
    tree = ts.first()
    print(f"Visualizing first tree (interval: {tree.interval})")
    print(f"Total trees in sequence: {ts.num_trees}")
    
    # Create a color map for populations
    node_colors = {}
    color_map = {0: 'red', 1: 'blue', 2: 'green', 3: 'purple'}
    
    for node in ts.nodes():
        if node.population in color_map:
            node_colors[node.id] = color_map[node.population]
    
    # Note: Tree visualization in Jupyter requires display capability
    # For basic text representation:
    print("\nTree structure (text representation):")
    print(tree.draw_text())

## Dxy Decomposition by Mutation Origin

This analysis decomposes the genetic divergence (Dxy) between populations 1 and 2 into two components:
1. **Ancestral polymorphism**: Mutations that existed in the ancestral population before the split
2. **New mutations**: Mutations that arose independently in each population after the split

We identify mutation origins by examining the time when each mutation arose relative to the split time.

In [ ]:
# Dxy decomposition analysis
print("Analyzing Dxy decomposition by mutation origin")
print("=" * 60)

# Get samples from populations 1 and 2
pop1_samples = pop_samples.get(1, [])
pop2_samples = pop_samples.get(2, [])

if len(pop1_samples) == 0 or len(pop2_samples) == 0:
    print("Warning: Need samples from both population 1 and 2")
else:
    # Calculate total Dxy between populations
    dxy_total = ts.divergence(sample_sets=[pop1_samples, pop2_samples], mode='site')
    dxy_total_value = dxy_total[0] if len(dxy_total) > 0 else 0
    
    print(f"Total Dxy between pop1 and pop2: {dxy_total_value:.6f}")
    print(f"Split time: {Tsplit} generations (forward time)")
    
    # Classify mutations by origin
    ancestral_mutations = []
    new_mutations = []
    
    # Iterate through all mutations
    for mut in ts.mutations():
        # Get the time of the mutation (in generations ago from present)
        # For tree sequences, time 0 is present, and time increases going back
        node = ts.node(mut.node)
        mut_time = node.time
        
        # The split occurred at generation Tsplit in forward time
        # In time-ago from present: split_time_ago = Tend - Tsplit
        split_time_ago = Tend - Tsplit
        
        # If mutation time is greater than split time, it's ancestral
        if mut_time > split_time_ago:
            ancestral_mutations.append(mut)
        else:
            new_mutations.append(mut)
    
    print(f"\nMutation classification:")
    print(f"  Ancestral (pre-split): {len(ancestral_mutations):,} mutations")
    print(f"  New (post-split): {len(new_mutations):,} mutations")
    print(f"  Total: {len(list(ts.mutations())):,} mutations")
    
    # Calculate Dxy contribution from each class
    # We need to count pairwise differences at sites with each class of mutations
    
    ancestral_dxy = 0
    new_dxy = 0
    
    # Create sets of site IDs for each mutation class
    ancestral_sites = set(mut.site for mut in ancestral_mutations)
    new_sites = set(mut.site for mut in new_mutations)
    
    # Calculate divergence for each site
    for site in ts.sites():
        # Get genotypes at this site
        genotypes = []
        for sample in pop1_samples + pop2_samples:
            # Find genotype for this sample at this site
            gt = 0
            for mut in site.mutations:
                # Check if this sample carries the mutation
                node = mut.node
                # Simple check: is the sample descended from this node?
                # This is simplified; proper implementation would trace tree
            genotypes.append(gt)
    
    # Alternative approach: use branch lengths
    # Calculate divergence using only ancestral polymorphisms
    print(f"\nDxy decomposition (approximate):")
    prop_ancestral = len(ancestral_mutations) / max(len(list(ts.mutations())), 1)
    prop_new = len(new_mutations) / max(len(list(ts.mutations())), 1)
    
    dxy_ancestral_est = dxy_total_value * prop_ancestral
    dxy_new_est = dxy_total_value * prop_new
    
    print(f"  Ancestral polymorphism contribution: {dxy_ancestral_est:.6f} ({prop_ancestral*100:.1f}%)")
    print(f"  New mutation contribution: {dxy_new_est:.6f} ({prop_new*100:.1f}%)")
    
    # Store results for plotting
    dxy_results = {
        'total': dxy_total_value,
        'ancestral': dxy_ancestral_est,
        'new': dxy_new_est,
        'split_time': Tsplit,
        'n_ancestral_muts': len(ancestral_mutations),
        'n_new_muts': len(new_mutations)
    }

## Visualize Dxy Decomposition

Create a figure showing the decomposition of Dxy into ancestral and new mutation components.

In [ ]:
# Improved Dxy decomposition calculation
print("Computing precise Dxy decomposition...")

if len(pop1_samples) > 0 and len(pop2_samples) > 0:
    # Get genotype matrix
    genotypes = ts.genotype_matrix()
    
    # Calculate split time in "time ago" coordinates
    split_time_ago = Tend - Tsplit
    
    # Classify each site by mutation origin
    ancestral_divergence = 0
    new_divergence = 0
    total_comparisons = len(pop1_samples) * len(pop2_samples)
    
    for site_idx, site in enumerate(ts.sites()):
        # Determine if this site has ancestral or new mutations
        is_ancestral = False
        for mut in site.mutations:
            node = ts.node(mut.node)
            if node.time > split_time_ago:
                is_ancestral = True
                break
        
        # Count pairwise differences at this site
        site_divergence = 0
        for i in pop1_samples:
            for j in pop2_samples:
                if genotypes[site_idx, i] != genotypes[site_idx, j]:
                    site_divergence += 1
        
        site_dxy = site_divergence / total_comparisons
        
        if is_ancestral:
            ancestral_divergence += site_dxy
        else:
            new_divergence += site_dxy
    
    # Normalize by sequence length
    ancestral_dxy = ancestral_divergence / ts.sequence_length
    new_dxy = new_divergence / ts.sequence_length
    total_dxy = ancestral_dxy + new_dxy
    
    print(f"\nPrecise Dxy decomposition:")
    print(f"  Total Dxy: {total_dxy:.6f}")
    print(f"  Ancestral contribution: {ancestral_dxy:.6f} ({ancestral_dxy/max(total_dxy, 1e-10)*100:.1f}%)")
    print(f"  New mutation contribution: {new_dxy:.6f} ({new_dxy/max(total_dxy, 1e-10)*100:.1f}%)")
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel 1: Bar plot of contributions
    ax = axes[0]
    categories = ['Ancestral\nPolymorphism', 'New\nMutations', 'Total']
    values = [ancestral_dxy, new_dxy, total_dxy]
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    
    bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Dxy', fontsize=12, fontweight='bold')
    ax.set_title(f'Dxy Decomposition\n(Split time = {Tsplit} generations)', 
                 fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Panel 2: Stacked bar showing proportions
    ax = axes[1]
    prop_ancestral = ancestral_dxy / max(total_dxy, 1e-10)
    prop_new = new_dxy / max(total_dxy, 1e-10)
    
    ax.bar(['Dxy Components'], [prop_ancestral], label='Ancestral', 
           color='#3498db', alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.bar(['Dxy Components'], [prop_new], bottom=[prop_ancestral], 
           label='New mutations', color='#e74c3c', alpha=0.7, 
           edgecolor='black', linewidth=1.5)
    
    ax.set_ylabel('Proportion of Dxy', fontsize=12, fontweight='bold')
    ax.set_title('Relative Contributions to Dxy', fontsize=13, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add percentage labels
    ax.text(0, prop_ancestral/2, f'{prop_ancestral*100:.1f}%', 
            ha='center', va='center', fontsize=12, fontweight='bold', color='white')
    ax.text(0, prop_ancestral + prop_new/2, f'{prop_new*100:.1f}%', 
            ha='center', va='center', fontsize=12, fontweight='bold', color='white')
    
    plt.tight_layout()
    plt.show()
    
    # Store results
    dxy_decomp_results = {
        'split_time': Tsplit,
        'total_dxy': total_dxy,
        'ancestral_dxy': ancestral_dxy,
        'new_dxy': new_dxy,
        'prop_ancestral': prop_ancestral,
        'prop_new': prop_new
    }
    
    print("\n✓ Visualization complete")
else:
    print("Cannot create visualization: need samples from both populations")

## Analyze Admixture Proportions

Examine the genetic ancestry of the admixed population by looking at the proportion of ancestry from each source population.

In [ ]:
# Analyze ancestry proportions in the admixed population (p3)
if 3 in pop_samples and len(pop_samples[3]) > 0:
    print("Admixture Analysis")
    print("=" * 60)
    
    admixed_samples = pop_samples[3]
    pop1_samples = pop_samples.get(1, [])
    pop2_samples = pop_samples.get(2, [])
    
    if len(pop1_samples) > 0 and len(pop2_samples) > 0:
        # Calculate genetic similarity of admixed population to each source
        # Using divergence as a proxy for ancestry
        
        dxy_admix_p1 = ts.divergence(sample_sets=[admixed_samples, pop1_samples], mode='site')
        dxy_admix_p2 = ts.divergence(sample_sets=[admixed_samples, pop2_samples], mode='site')
        dxy_p1_p2 = ts.divergence(sample_sets=[pop1_samples, pop2_samples], mode='site')
        
        div_admix_p1 = dxy_admix_p1[0] if len(dxy_admix_p1) > 0 else 0
        div_admix_p2 = dxy_admix_p2[0] if len(dxy_admix_p2) > 0 else 0
        div_p1_p2 = dxy_p1_p2[0] if len(dxy_p1_p2) > 0 else 0
        
        print(f"Divergence between admixed pop and Pop1: {div_admix_p1:.6f}")
        print(f"Divergence between admixed pop and Pop2: {div_admix_p2:.6f}")
        print(f"Divergence between Pop1 and Pop2: {div_p1_p2:.6f}")
        
        # Estimate admixture proportion using divergence
        # Expected: div(admix, p1) ≈ (1-f) * div(p1, p2)
        # Expected: div(admix, p2) ≈ f * div(p1, p2)
        if div_p1_p2 > 0:
            est_f = 1 - (div_admix_p1 / div_p1_p2)
            print(f"\nExpected admixture fraction from Pop1: {f:.3f}")
            print(f"Estimated admixture fraction from Pop1: {est_f:.3f}")
            
            # Visualize
            fig, ax = plt.subplots(figsize=(8, 5))
            categories = ['Expected', 'Estimated']
            values = [f, est_f]
            colors = ['lightblue', 'orange']
            
            ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
            ax.set_ylabel('Admixture Fraction from Pop1')
            ax.set_title('Admixture Proportion Comparison')
            ax.set_ylim([0, 1])
            ax.axhline(y=f, color='red', linestyle='--', alpha=0.5, label=f'Expected (f={f})')
            ax.legend()
            
            # Add value labels on bars
            for i, v in enumerate(values):
                ax.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
            
            plt.tight_layout()
            plt.show()
else:
    print("No admixed population samples found for analysis.")

## Summary Statistics Table

Compile all key statistics into a summary table.

In [ ]:
# Create comprehensive summary
print("Simulation Summary")
print("=" * 80)

summary_data = []

for pop_id in range(ts.num_populations):
    samples = pop_samples.get(pop_id, [])
    if len(samples) >= 2:
        pi = ts.diversity(sample_sets=[samples], mode='site')
        pi_value = pi[0] if len(pi) > 0 else np.nan
        
        pop_name = f"Pop{pop_id}"
        if pop_id == 0:
            pop_name = "Ancestral (p0)"
        elif pop_id == 1:
            pop_name = "Population 1 (p1)"
        elif pop_id == 2:
            pop_name = "Population 2 (p2)"
        elif pop_id == 3:
            pop_name = "Admixed (p3)"
        
        summary_data.append({
            'Population': pop_name,
            'ID': pop_id,
            'Samples': len(samples),
            'Diversity (π)': f"{pi_value:.6f}" if not np.isnan(pi_value) else "N/A"
        })

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

print("\n" + "=" * 80)
print("Simulation completed successfully!")
print(f"Tree sequence saved to: {output_trees}")
print(f"SLiM script saved to: {slim_script_file}")

## Cleanup (Optional)

Remove temporary files if desired.

In [ ]:
# Uncomment to remove output files
# import os
# if os.path.exists(output_trees):
#     os.remove(output_trees)
#     print(f"Removed {output_trees}")
# if os.path.exists(slim_script_file):
#     os.remove(slim_script_file)
#     print(f"Removed {slim_script_file}")

print("Files retained for further analysis.")